<a href="https://colab.research.google.com/github/rajaraomis/Retail-Data-Analytics/blob/main/Multiple%2BLinear%2BRegression%2B_%2Bhousing%2Bcase%2Bstudy(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Import Numpy as np
import pandas as pd

In [ ]:
%%bash
pip install pandas numpy scikit-learn statsmodels matplotlib seaborn

In [ ]:
import pandas as pd
import numpy as np
url = "https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/insurance.csv"
df = pd.read_csv(url)
print(df.shape)
df.head()

In [ ]:
df.info()
df.isnull().sum() # check for missing values

In [ ]:
df.describe()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error
# quick one-variable baseline
X_slr = df[["age"]]
y = df["charges"]
X_train, X_test, y_train, y_test = train_test_split(X_slr, y, test_size=0.2, random_state=42)
slr = LinearRegression().fit(X_train, y_train)
pred = slr.predict(X_test)
print("SLR (age only) Test R2:"
, r2_score(y_test, pred))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
# correlation among numeric variables
numeric_cols = ["age", "bmi", "children", "charges"]
corr = df[numeric_cols].corr()
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Matrix — Numeric Features")
plt.show()

In [ ]:
# does smoking status matter?
sns.boxplot(x=
"smoker"
, y=
"charges"
, data=df)
plt.title("Charges by Smoking Status")
plt.show()

In [ ]:
df_encoded = pd.get_dummies(
df,
columns=["sex"
,
"smoker"
,
"region"],
drop_first=True # avoids the dummy variable trap
)
df_encoded.head()

In [ ]:
features = [
    "age", "bmi", "children", "charges",
    "sex_male", "smoker_yes",
    "region_northwest", "region_southeast", "region_southwest"
]
print(features)

In [ ]:
# convert boolean dummy columns to integers (0/1) for cleanliness
bool_cols = df_encoded.select_dtypes(include=
"bool").columns
df_encoded[bool_cols] = df_encoded[bool_cols].astype(int)

In [ ]:
X = df_encoded.drop(columns=["charges"])
y = df_encoded["charges"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
mlr = LinearRegression()
mlr.fit(X_train, y_train)
# view the coefficients alongside feature names
coef_table = pd.DataFrame({
"feature": X.columns,
"coefficient": mlr.coef_
})
print(coef_table)
print("Intercept:"
, mlr.intercept_)

In [ ]:
y_pred = mlr.predict(X_test)
r2 = r2_score(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred) ** 0.5
print("Test R2: "
, round(r2, 4))
print("Test RMSE:"
, round(rmse, 2))

In [ ]:
import statsmodels.api as sm
X_train_sm = sm.add_constant(X_train.astype(float))
ols_model = sm.OLS(y_train.astype(float), X_train_sm).fit()
print(ols_model.summary())

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
X_vif = sm.add_constant(X.astype(float))
vif_data = pd.DataFrame()
vif_data["feature"] = X_vif.columns
vif_data["VIF"] = [
variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])
]
print(vif_data)

In [ ]:
def evaluate(feature_list, label):
    Xtr, Xte = X_train[feature_list], X_test[feature_list]
    model = LinearRegression().fit(Xtr, y_train)
    preds = model.predict(Xte)
    r2 = r2_score(y_test, preds)
    rmse = mean_squared_error(y_test, preds) ** 0.5
    print(f"{label:35s} Test R2 = {r2:.4f} RMSE = {rmse:.2f}")

full_features = list(X.columns)
reduced_features = ["age", "bmi", "children", "smoker_yes"]
evaluate(full_features, "Full model (8 features)")
evaluate(reduced_features, "Reduced model (4 features)")
evaluate(["smoker_yes"], "Baseline (smoker only)")
evaluate(["age"], "SLR (age only)")

In [ ]:
final_features = ["age"
,
"bmi"
,
"children"
,
"smoker_yes"]
final_model = LinearRegression().fit(X_train[final_features], y_train)
final_coefs = pd.DataFrame({
"feature": final_features,
"coefficient": final_model.coef_
})
print(final_coefs)
print("Intercept:"
, final_model.intercept_)

In [ ]:
df["age_squared"] = df["age"] ** 2

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso
import pandas as pd

# Train models
lr = LinearRegression()
ridge = Ridge(alpha=1.0)
lasso = Lasso(alpha=0.1)

lr.fit(X_train, y_train)
ridge.fit(X_train, y_train)
lasso.fit(X_train, y_train)

# Compare coefficients
coef_df = pd.DataFrame({
    "Feature": X_train.columns,
    "LinearRegression": lr.coef_,
    "Ridge": ridge.coef_,
    "Lasso": lasso.coef_
})

# Show sex_male and region dummy variables
coef_df[coef_df["Feature"].str.contains("sex_male|region")]

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

selected_features = []
remaining_features = list(X_train.columns)

best_r2 = -float("inf")

while remaining_features:
    scores = []

    for feature in remaining_features:
        features = selected_features + [feature]

        model = LinearRegression()
        model.fit(X_train[features], y_train)

        pred = model.predict(X_test[features])
        r2 = r2_score(y_test, pred)

        scores.append((feature, r2))

    # Select feature giving highest R²
    best_feature, best_score = max(scores, key=lambda x: x[1])

    if best_score > best_r2:
        best_r2 = best_score
        selected_features.append(best_feature)
        remaining_features.remove(best_feature)
        print(f"Added: {best_feature}, Test R² = {best_score:.4f}")
    else:
        break

print("\nSelected Features:")
print(selected_features)

In [ ]:
import matplotlib.pyplot as plt

# Predict on test data
y_pred = final_model.predict(X_test[final_features])

# Calculate residuals
residuals = y_test - y_pred

# Residual Plot
plt.figure(figsize=(8,5))
plt.scatter(y_pred, residuals, alpha=0.7)
plt.axhline(y=0, color='red', linestyle='--')
plt.xlabel("Predicted Charges")
plt.ylabel("Residuals (Actual - Predicted)")
plt.title("Residuals vs Predicted Values")
plt.show()